# 00 · Background, reading list, and what is new here

**Read this first.** It assumes you know what a neural network, a loss and backpropagation are, and nothing about
quantization. It does three things:

1. explains the vocabulary used in all the other notebooks;
2. gives a reading list, ordered from "start here" to "for later";
3. tells you, for every technique in this tutorial, whether it is **common practice** (you can read about it in the
   literature and should treat it as a baseline), **known but less common**, or **new or modified in vvtk_quant**
   (even if only a small change). Keep this distinction clear when you read papers and when you write your report.

### The three tags used in every notebook

> 📘 **Common practice**: standard, used in most papers and tools. Read the reference; do not present it as ours.

> 📗 **Known, less common**: published, but not the default. Cite it; it is the closest competitor to our ideas.

> 🧪 **New or modified in vvtk_quant**: something this library does differently, sometimes a small change. These are
> the things your study tests. A tag 🧪 does **not** mean "nobody ever did it": the literature search is not finished.
> It means "not the standard recipe; check before claiming".

## 1 · What is quantization, and why

A trained network stores its weights as 32-bit (fp32) or 16-bit (bf16/fp16) floats. **Quantizing** a weight means
replacing it with one of a small set of values, the **levels** $l_0 < \dots < l_{L-1}$. With $L = 2^b$ levels we only
need to store a $b$-bit **index** per weight, plus the small table of levels.

Why: memory and bandwidth. A model with $N$ weights takes $32N$ bits in fp32 and about $bN$ bits at $b$ bits. On
phones, robots, CPUs and FPGAs (the BRAINS project targets), memory and memory traffic dominate cost and energy.

| format | bits/weight | 3 M-param model | 0.6 B-param model (Qwen3-0.6B size) |
|---|---|---|---|
| fp32 | 32 | 12 MB | 2.4 GB |
| bf16 | 16 | 6 MB | 1.2 GB |
| 8-bit | 8 | 3 MB | 0.6 GB |
| 4-bit | 4 | 1.5 MB | 0.3 GB |
| 2-bit | 2 | 0.75 MB | 0.15 GB |

Quantization adds an error $w - Q(w)$ to every weight. At 8 bits it is almost harmless; at 4 bits it needs care; at
2–3 bits it destroys a network unless we **train** with the quantizer in the loop. This tutorial is about weights only;
quantizing activations is a further step.

In [1]:
# The table above, computed (bytes = bits / 8)
for name, n in [("3 M", 3e6), ("0.6 B", 0.6e9)]:
    print(name, {b: f"{b * n / 8 / 1e6:.2f} MB" for b in [32, 16, 8, 4, 2]})

3 M {32: '12.00 MB', 16: '6.00 MB', 8: '3.00 MB', 4: '1.50 MB', 2: '0.75 MB'}
0.6 B {32: '2400.00 MB', 16: '1200.00 MB', 8: '600.00 MB', 4: '300.00 MB', 2: '150.00 MB'}


## 2 · Vocabulary

| term | meaning |
|---|---|
| **level**, **grid** | the allowed values $l_k$; a *uniform* grid has equal spacing, a *non-uniform* one does not |
| **index**, **code** | the integer $k$ stored for a weight; $b$ bits each |
| **scale** | the factor that maps a normalised grid (e.g. integers $-7\dots7$) to the weight range; per-tensor, per-channel or per-block |
| **bin**, **limit** | the interval of weights sent to the same level; the boundaries $m_i$ between bins |
| **PTQ** (post-training quantization) | quantize a trained model, no training (maybe a little calibration data) |
| **QAT** (quantization-aware training) | train (or fine-tune) with the quantizer in the forward pass |
| **latent / master weights** | the float weights kept during QAT; the deployed weights are $Q(w)$ |
| **STE** (straight-through estimator) | the standard QAT trick: forward $Q(w)$, backward as if $Q$ were the identity |
| **hard / soft** staircase | the true quantizer $Q$ / a smooth approximation $Q_T$ with temperature $T$ |
| **anchor** | a loss term that pulls each latent weight to its level (in vvtk_quant) |
| **freezing** | fixing a weight to its level for the rest of training |
| **oscillation** | a latent weight crossing a limit back and forth, so the deployed value keeps flipping |
| **MSE**, **SQNR** | mean squared quantization error; signal-to-quantization-noise ratio in dB |
| **VQ** (vector quantization) | quantize groups of $d$ weights jointly to codewords in $\mathbb{R}^d$; scalar quantization is $d=1$ |

## 3 · Reading list

**Start here (in this order, the first two are enough for week 1):**

1. M. Nagel et al., *A White Paper on Neural Network Quantization*, 2021, arXiv:2106.08295. Practical, clear: PTQ,
   QAT, STE, per-channel scales, the standard pipelines.
2. A. Gholami et al., *A Survey of Quantization Methods for Efficient Neural Network Inference*, 2021, arXiv:2103.13630.
   Broad map of the field: uniform vs non-uniform, symmetric vs asymmetric, PTQ vs QAT, mixed precision.
3. R. Krishnamoorthi, *Quantizing deep convolutional networks for efficient inference: A whitepaper*, 2018,
   arXiv:1806.08342. The 8-bit integer recipe used in TensorFlow Lite.
4. B. Jacob et al., *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference*,
   CVPR 2018, arXiv:1712.05877. Where "fake quantization" QAT comes from.

**The STE and its baselines:**

5. Y. Bengio, N. Léonard, A. Courville, *Estimating or Propagating Gradients Through Stochastic Neurons for Conditional
   Computation*, 2013, arXiv:1308.3432. The STE.
6. M. Courbariaux, Y. Bengio, J.-P. David, *BinaryConnect*, NeurIPS 2015, arXiv:1511.00363. Latent weights + projection.
7. P. Yin et al., *Understanding Straight-Through Estimator in Training Activation Quantized Neural Nets*, ICLR 2019,
   arXiv:1903.05662. Why STE works and when it does not.
8. S. K. Esser et al., *Learned Step Size Quantization* (LSQ), ICLR 2020, arXiv:1902.08153. The strong uniform STE
   baseline: learn the scale.
9. S. Ma et al., *The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits* (BitNet b1.58), 2024,
   arXiv:2402.17764. Ternary QAT with STE and an absmean scale; the modern extreme case.

**Non-uniform levels and classic quantization theory:**

10. S. P. Lloyd, *Least squares quantization in PCM*, IEEE Trans. Inf. Theory 28(2):129–137, 1982; J. Max, *Quantizing
    for minimum distortion*, IRE Trans. Inf. Theory IT-6:7–12, 1960. The Lloyd-Max quantizer.
11. R. M. Gray, D. L. Neuhoff, *Quantization*, IEEE Trans. Inf. Theory 44(6):2325, 1998. The history and theory,
    including Bennett's high-resolution analysis and the $p^{1/3}$ level density used in notebook 02.
12. S. Han, H. Mao, W. J. Dally, *Deep Compression*, ICLR 2016, arXiv:1510.00149. k-means (Lloyd) weight sharing, then
    fine-tuning: non-uniform levels in deep learning.
13. T. Dettmers et al., *QLoRA*, NeurIPS 2023, arXiv:2305.14314. NF4: levels at the quantiles of a normal distribution
    (an equal-probability grid), now standard for 4-bit LLM fine-tuning.
14. S. Kim et al., *SqueezeLLM: Dense-and-Sparse Quantization*, ICML 2024, arXiv:2306.07629. Sensitivity-weighted
    k-means levels for LLM PTQ.
15. E. Frantar et al., *GPTQ*, ICLR 2023, arXiv:2210.17323. The standard LLM PTQ method (uniform grid, second-order
    error compensation).
16. B. Darvish Rouhani et al., *Microscaling Data Formats for Deep Learning*, 2023, arXiv:2310.10537. FP8/FP6/FP4 with
    shared block scales (MX formats), the hardware direction.
17. V. Egiazarian et al., *Extreme Compression of LLMs via Additive Quantization* (AQLM), ICML 2024, arXiv:2401.06118.
    Vector quantization (codebooks) at 2 bits.

**Closest to the ideas in this tutorial (read after notebook 04):**

18. R. Gong et al., *Differentiable Soft Quantization* (DSQ), ICCV 2019, arXiv:1908.05033. A soft staircase with a
    learnable temperature, on a uniform grid.
19. E. Agustsson et al., *Soft-to-Hard Vector Quantization*, NeurIPS 2017, arXiv:1704.00648. Soft quantization
    annealed to hard, for compression.
20. C. Louizos et al., *Relaxed Quantization for Discretized Neural Networks*, ICLR 2019, arXiv:1810.01875.
21. Y. Bai, Y.-X. Wang, E. Liberty, *ProxQuant*, ICLR 2019, arXiv:1810.00861. A proximal pull towards the grid
    (close to our anchor).
22. M. Nagel et al., *Overcoming Oscillations in Quantization-Aware Training*, ICML 2022, arXiv:2203.11086.
    Oscillations, dampening, and freezing of oscillating weights (close to our freezing).

For notebook 07: A. Graves et al., *Connectionist Temporal Classification*, ICML 2006, doi:10.1145/1143844.1143891.

## 4 · The map: what is common, what is less common, what is new here

| technique | tag | where in the tutorial | read |
|---|---|---|---|
| uniform int-$b$ grid, per-tensor absmax scale | 📘 common | 02, 03, 04 (`uniform`) | 1, 2, 3 |
| absmean scale, ternary $\{-a,0,a\}$ | 📘 common | 02 (exercise) | 9 |
| PTQ: quantize a trained model, no training | 📘 common | 03 | 1, 15 |
| QAT with STE and latent float weights | 📘 common | 01, 04 (`ste-*`) | 5, 6, 7 |
| learned scale (LSQ) | 📘 common baseline | 04 (exercise) | 8 |
| non-uniform levels by k-means / Lloyd-Max | 📘 common (PTQ and compression) | 02, 03 (`lloyd`) | 10, 12, 14 |
| quantile (equal-probability) levels | 📘 common (NF4 uses normal quantiles) | 02 (`eqprob`) | 13 |
| FP4 / FP8 value tables with a scale | 📘 common | 02 (`fp`) | 16 |
| packed $b$-bit indices + level table on disk | 📘 common | 03 | 12 |
| evaluating the **hard** model | 📘 common (mandatory) | everywhere | 1 |
| soft (sigmoid) staircase with temperature annealing instead of STE | 📗 known, less common | 01, 04 (`soft-*`) | 18, 19 |
| a loss/prox term pulling weights to the grid | 📗 known, less common | 05 | 21 |
| freezing weights during QAT | 📗 known, less common (for oscillating weights) | 05 | 22 |
| companding: level density $\propto p^{1/3}$ | 📗 classic theory, rarely used in deep learning | 02 | 11 |
| vector quantization (k-means codebooks over groups of $ weights) | 📗 known (LLM 2-bit: AQLM) | 08 (starter) | 17, 11 |
| soft assignment to codewords with annealed $\tau$ | 📗 known (soft-to-hard VQ) | 08 | 19 |
| anchor to the nearest codeword, VQ + freezing | 🧪 vvtk_quant ideas carried to VQ | 08 (menu) | — |
| **empirical** quantiles of each tensor (not of a normal) as levels | 🧪 small modification of NF4 | 02, 03 (`eqprob`) | compare with 13 |
| soft staircase on **non-uniform** (Lloyd) levels | 🧪 modified (DSQ is uniform) | 01, 04 | compare with 18 |
| temperature **relative to each local step** $T/\Delta_i$ | 🧪 small modification | 01 Step 5 | — |
| STE with **Lloyd** levels as a control (separates "where are the levels" from "how the gradient flows") | 🧪 experimental design | 04, 06, 07 (`ste-lloyd`) | — |
| anchor in units of the local step, $((w-Q(w))/\Delta)^2$, usable with STE too | 🧪 modified | 05 | compare with 21 |
| **fused anchor**: anchor gradient added inside the layer's backward, nothing saved but $w$ | 🧪 new (implementation) | 05 Step 2 | recompute-in-backward is standard (checkpointing) |
| freezing the **stable** weights (distance to level), aiming at dropping their optimizer state | 🧪 new direction | 05 | compare with 22 |
| anchor schedule (ramp) coupled to the temperature schedule | 🧪 new (hyperparameter) | 05 | — |

When you write: everything 📘 goes in "background/baselines", 📗 in "related work" (and, ideally, in your result
tables as competitors), 🧪 in "what we try". If you find a paper that already does a 🧪 item, tell Antonio: that is a
useful result, not a failure.

## 5 · How the notebooks are organised

| notebook | question | runtime |
|---|---|---|
| 01 · the staircase | what is STE, what is the soft staircase, how do their gradients differ? | seconds |
| 02 · the levels | where to put the levels for minimum error? | seconds |
| 03 · PTQ on CIFAR | how much does a trained CNN lose without training? | minutes |
| 04 · QAT soft vs STE | under a fair budget, which training recipe recovers best? | ~15 min |
| 05 · anchors and freezing | the research ideas: anchor, fused anchor (memory), freezing | ~10 min |
| 06 · tiny LM | the same comparison on a character language model (your LM starter) | ~5 min |
| 07 · phoneme decoder | the same on TIMIT CTC phoneme recognition (your speech starter) | ~10 min |
| 08 · vector quantization | from scalar levels to codebooks: VQ PTQ and QAT on the tiny LM, a menu of VQ variants | ~10 min |

Every experiment in 03–07 is **one seed** and a short schedule: smoke tests that show the method runs and what to
look at, not results. Your first job is to make them solid (3 seeds, longer schedules), then extend them.

**Next:** [01 · The staircase](01_staircase.ipynb).